# 05 平均數推論分析 — Mean Analysis (Two-Sample T-Test)
**Research Question: Gender and Current Alcohol Use Frequency**

- **Group variable:** `WhatIsYourSex` (1 = Male, 2 = Female)
- **Response variable:** `CurrentAlcoholUse` (原始 1–7 連續值，數字越大代表飲酒頻率越高)
- **Method:** Two-Sample T-Test (Welch's)

## 1. 載入套件與資料 | Load Libraries & Data

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt

PROJECT = r'C:\Users\88690\OneDrive\桌面\project-cycle-3-main'

df = pd.read_csv(f'{PROJECT}/data/Processed/YRBS_2007_cleaned.csv')
print('資料維度 Shape:', df.shape)
df[['WhatIsYourSex', 'CurrentAlcoholUse']].head(10)

## 2. 資料清理 | Data Cleaning

使用 `CurrentAlcoholUse` 原始 1–7 數值作為連續變數：
- 1 = 沒有喝酒
- 2 = 1–2 天
- 3 = 3–5 天
- 4 = 6–9 天
- 5 = 10–19 天
- 6 = 20–29 天
- 7 = 所有 30 天

In [ ]:
df_clean = df[['WhatIsYourSex', 'CurrentAlcoholUse']].dropna()
df_clean = df_clean[df_clean['WhatIsYourSex'].isin([1.0, 2.0])]
df_clean['sex_label'] = df_clean['WhatIsYourSex'].map({1.0: 'Male', 2.0: 'Female'})

print('清理後資料維度:', df_clean.shape)
print()
print('CurrentAlcoholUse 分佈:')
print(df_clean['CurrentAlcoholUse'].value_counts().sort_index())

## 3. 描述統計摘要 | Descriptive Summary

In [ ]:
male_alc   = df_clean[df_clean['sex_label'] == 'Male']['CurrentAlcoholUse']
female_alc = df_clean[df_clean['sex_label'] == 'Female']['CurrentAlcoholUse']

summary = pd.DataFrame({
    'n':      [len(male_alc),           len(female_alc)],
    'mean':   [male_alc.mean().round(4), female_alc.mean().round(4)],
    'std':    [male_alc.std().round(4),  female_alc.std().round(4)],
    'median': [male_alc.median(),        female_alc.median()],
    'min':    [male_alc.min(),           female_alc.min()],
    'max':    [male_alc.max(),           female_alc.max()]
}, index=['Male', 'Female'])

print('=== 描述統計摘要 ===')
print(summary.to_string())
print()
print(f'Difference (Male - Female): {(male_alc.mean() - female_alc.mean()):.4f}')

## 4. 假設設定 | Hypotheses

$$H_0: \mu_{male} - \mu_{female} = 0$$
$$H_1: \mu_{male} - \mu_{female} \neq 0$$

- **Test type:** Two-sided
- **Significance level:** $\alpha = 0.05$
- **Method:** Welch's Two-Sample T-Test（不假設兩組變異數相等）

## 5. Two-Sample T-Test

In [ ]:
t_stat, p_value = stats.ttest_ind(male_alc, female_alc, equal_var=False)

print('=== Welch Two-Sample T-Test 結果 ===')
print(f'T-statistic : {t_stat:.4f}')
print(f'P-value     : {p_value:.4f}')
print()
if p_value < 0.05:
    print('結論: p-value < 0.05 → Reject H₀')
    print('      兩組飲酒頻率平均值有顯著差異 (statistically significant)')
else:
    print('結論: p-value ≥ 0.05 → Fail to reject H₀')
    print('      沒有足夠證據顯示兩組飲酒頻率平均值有差異')

## 6. 95% 信賴區間 | Confidence Interval

In [ ]:
diff = male_alc.mean() - female_alc.mean()
se   = np.sqrt(male_alc.var()/len(male_alc) + female_alc.var()/len(female_alc))
ci_lower = diff - 1.96 * se
ci_upper = diff + 1.96 * se

print('=== 95% 信賴區間 (Male - Female) ===')
print(f'Difference : {diff:.4f}')
print(f'SE         : {se:.4f}')
print(f'95% CI     : ({ci_lower:.4f}, {ci_upper:.4f})')

## 7. 視覺化 | Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
colors = ['#4C72B0', '#DD8452']

# --- 7a. Bar chart 平均值比較 ---
ax = axes[0]
means = [male_alc.mean(), female_alc.mean()]
stds  = [male_alc.std(),  female_alc.std()]
bars = ax.bar(['Male', 'Female'], means, color=colors,
              width=0.5, edgecolor='white', linewidth=1.5,
              yerr=stds, capsize=6, error_kw={'linewidth': 1.5})
for bar, mean in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.08,
            f'{mean:.4f}',
            ha='center', va='bottom', fontsize=12, fontweight='bold')
ax.set_ylim(0, max(means) * 1.4)
ax.set_ylabel('Mean CurrentAlcoholUse Score', fontsize=11)
ax.set_xlabel('Sex', fontsize=11)
ax.set_title('Mean Alcohol Use Frequency by Sex\n(YRBS 2007)', fontsize=13, fontweight='bold')
ax.spines[['top', 'right']].set_visible(False)

# --- 7b. 信賴區間圖 ---
ax = axes[1]
ax.errorbar(x=0, y=diff, yerr=[[diff - ci_lower], [ci_upper - diff]],
            fmt='o', color='#2d6a4f', markersize=10,
            capsize=8, capthick=2, linewidth=2, label='95% CI')
ax.axhline(0, color='red', linestyle='--', linewidth=1.5, label='H₀: diff = 0')
ax.set_xlim(-0.5, 0.5)
ax.set_xticks([0])
ax.set_xticklabels(['Male − Female'])
ax.set_ylabel('Difference in Means', fontsize=11)
ax.set_title('95% CI for Difference\nin Means (Male − Female)', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.spines[['top', 'right']].set_visible(False)
ax.annotate(f'T = {t_stat:.3f}\np = {p_value:.4f}',
            xy=(0.05, 0.85), xycoords='axes fraction',
            fontsize=11, color='#333333',
            bbox=dict(boxstyle='round,pad=0.3', fc='#f0f0f0', ec='grey'))

plt.tight_layout()
plt.savefig(f'{PROJECT}/outputs/figures/mean_inference_alcohol_by_sex.png', dpi=150, bbox_inches='tight')
plt.show()
print('圖表已儲存至 outputs/figures/mean_inference_alcohol_by_sex.png')

## 8. 儲存結果 | Save Results

In [ ]:
results = pd.DataFrame({
    'Group':       ['Male', 'Female'],
    'n':           [len(male_alc), len(female_alc)],
    'Mean':        [round(male_alc.mean(), 4), round(female_alc.mean(), 4)],
    'Std':         [round(male_alc.std(), 4),  round(female_alc.std(), 4)],
    'T_statistic': [round(t_stat, 4), ''],
    'P_value':     [round(p_value, 4), ''],
    'CI_lower':    [round(ci_lower, 4), ''],
    'CI_upper':    [round(ci_upper, 4), '']
})
results.to_csv(f'{PROJECT}/outputs/tables/mean_inference_alcohol_by_sex.csv', index=False)
print('結果已儲存至 outputs/tables/mean_inference_alcohol_by_sex.csv')
print()
print('=== 05 平均數推論分析完成 ===')